In [ ]:
from Bio import Entrez
import re
import json
from typing import Dict, List, Tuple, Optional
import time
import os

# Species to assembly mapping - add your species here
ASSEMBLY_MAPPING = {
    's_cerevisiae': 'GCF_000146045.2',
    's_pombe': 'GCF_000002945.2',
    'c_albicans': 'GCF_000182965.3',
}

# Read YAML configuration file and let it override the settings
# defined above, if it exists
import yaml

yaml_file = 'novabrowse_config.yaml'
if os.path.exists(yaml_file):
    with open(yaml_file, 'r') as f:
        yaml_config = yaml.safe_load(f)
        # Pick out the Entrez email from YAML if defined
        if 'entrez_email' in yaml_config:
            entrez_email = yaml_config['entrez_email']
        # The ASSEMBLY_MAPPING section contains one subsection per species.
        # Each subsection contains 'assembly_acc', 'fasta_file', and 'fasta_file_type' keys (nucl or prot).
        if 'ASSEMBLY_MAPPING' in yaml_config:
            # First, clear the default ASSEMBLY_MAPPING, then populate it with values from YAML.
            ASSEMBLY_MAPPING.clear()
            for species, config in yaml_config['ASSEMBLY_MAPPING'].items():
                if 'assembly_acc' in config:
                    ASSEMBLY_MAPPING[species] = config['assembly_acc']
else:
    print(f"YAML configuration file '{yaml_file}' not found. Using default assembly mappings.")

# Set your email here (required by NCBI)

# This defaults to the YAML configuration if the ENTREZ_EMAIL_ENV
# environment variable is not set.
Entrez.email = os.environ.get("ENTREZ_EMAIL_ENV", entrez_email)

# Refuse to run if Entrez.email is not set, since NCBI requires it for API access
if not Entrez.email:
    raise ValueError("Entrez email is not set. Please set the ENTREZ_EMAIL_ENV environment variable or define 'entrez_email' in the YAML configuration.")

def natural_sort_key(s):
    """
    Returns a tuple for consistent sorting of different types of chromosome identifiers.
    First element indicates type priority (number < letter < roman numeral)
    Second element is the actual sort value
    """
    try:
        # For numeric chromosomes, return (0, number) for highest priority
        num = int(s)
        return (0, num)
    except:
        if isinstance(s, str):
            # Check for roman numerals
            roman_nums = ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X', 'XI', 'XII', 'XIII', 'XIV', 'XV', 'XVI']
            if s.upper() in roman_nums:
                # Roman numerals get lowest priority
                return (2, roman_nums.index(s.upper()))
            # Letters and other strings get middle priority
            return (1, s)
        # Any other type gets lowest priority
        return (3, str(s))

def fetch_chromosome_info(assembly_accession: str) -> List[List[Tuple[str, int, str]]]:
    """Fetch chromosome information using assembly accession."""
    try:
        # Search nuccore for RefSeq sequences belonging to this assembly
        handle = Entrez.esearch(db="nuccore", term=f"{assembly_accession}[Assembly] AND refseq[filter]", retmax=500)
        record = Entrez.read(handle)
        handle.close()
        
        seq_ids = record['IdList']
        
        if not seq_ids:
            print(f"No RefSeq sequences found for {assembly_accession}")
            return []
        
        time.sleep(0.5)
        
        # Fetch sequence details
        handle = Entrez.esummary(db="nuccore", id=",".join(seq_ids))
        sequences = Entrez.read(handle)
        handle.close()
        
        # Dictionary to store chromosome parts
        chr_parts = {}
        
        # Process sequence information
        for seq in sequences:
            title = seq['Title']
            length = int(seq['Length'])
            accession = seq['AccessionVersion']
            extra = seq.get('Extra', '')
            
            # Skip if not an NC_ sequence
            if not accession.startswith('NC_'):
                continue
            
            # Try multiple patterns to extract chromosome and part information
            
            # Pattern for lungfish "part" format: "chromosome 1.part0"
            lungfish_part_match = re.search(r'chromosome[:\s]+(\d+)\.part(\d+)', title, re.IGNORECASE)
            if lungfish_part_match:
                chr_num = lungfish_part_match.group(1)
                part_num = lungfish_part_match.group(2)
                
                if chr_num not in chr_parts:
                    chr_parts[chr_num] = {}
                
                chr_parts[chr_num][f"part{part_num}"] = (accession, length, f"{chr_num}.part{part_num}")
                continue
            
            # Pattern for pleurodeles format: "chromosome 1_2"
            pleurodeles_match = re.search(r'chromosome[:\s]+(\d+)_(\d+)', title, re.IGNORECASE)
            if pleurodeles_match:
                chr_num = pleurodeles_match.group(1)
                part_num = pleurodeles_match.group(2)
                
                if chr_num not in chr_parts:
                    chr_parts[chr_num] = {}
                
                chr_parts[chr_num][part_num] = (accession, length, f"{chr_num}_{part_num}")
                continue
            
            # Try to get chromosome info from Extra field (p/q arms)
            chr_meta_match = re.search(r'chr(\d+)([pq])?', extra)
            if chr_meta_match:
                chr_num = chr_meta_match.group(1)
                arm = chr_meta_match.group(2) or ''
                
                if chr_num not in chr_parts:
                    chr_parts[chr_num] = {}
                
                if arm:
                    chr_parts[chr_num][arm] = (accession, length, f"{chr_num}{arm}")
                else:
                    chr_parts[chr_num]['whole'] = (accession, length, chr_num)
                continue
            
            # Standard chromosome pattern - handles both "chromosome 1" and "chromosome: 1"
            std_chr_match = re.search(r'chromosome[:\s]+(\d+|[XYZWMTIVp-q]+)(?![_\d])', title, re.IGNORECASE)
            if std_chr_match:
                chr_num = std_chr_match.group(1)
                
                if chr_num not in chr_parts:
                    chr_parts[chr_num] = {}
                
                chr_parts[chr_num]['whole'] = (accession, length, chr_num)
        
        # Convert to final format
        chromosomes = []
        
        # Sort chromosome numbers using natural_sort_key
        for chr_num in sorted(chr_parts.keys(), key=natural_sort_key):
            parts = chr_parts[chr_num]
            
            # For chromosomes with numeric parts (Pleurodeles: 1_1, 1_2)
            numeric_parts = [k for k in parts.keys() if k.isdigit()]
            if numeric_parts:
                chromosomes.append([parts[k] for k in sorted(numeric_parts, key=int)])
            
            # For chromosomes with "part" keys (Lungfish: part0, part1)
            elif any(k.startswith('part') for k in parts):
                part_keys = sorted([k for k in parts.keys() if k.startswith('part')], 
                                  key=lambda x: int(x[4:]))
                chromosomes.append([parts[k] for k in part_keys])
            
            # For chromosomes with p/q arms
            elif 'p' in parts and 'q' in parts:
                chromosomes.append([parts['p'], parts['q']])
            
            # For chromosomes with whole parts
            elif 'whole' in parts:
                chromosomes.append([parts['whole']])
            
            # For any other parts
            else:
                chromosomes.append([parts[next(iter(parts))]])
        
        return chromosomes
    
    except Exception as e:
        print(f"Error fetching chromosome info for {assembly_accession}: {str(e)}")
        return []

def generate_chromosome_data(species_list: List[str]) -> Dict[str, List]:
    """Generate chromosome information for all species and return as a dictionary."""
    chromosome_data = {}
    
    for species in species_list:
        assembly_id = ASSEMBLY_MAPPING.get(species)
        if not assembly_id:
            print(f"No assembly mapping found for {species}")
            continue
        
        print(f"Fetching data for {species}...")
        chromosomes = fetch_chromosome_info(assembly_id)
        
        if chromosomes:
            chromosome_data[species] = {
                'assembly_id': assembly_id,
                'chromosomes': [
                    {
                        'parts': [
                            {
                                'accession': part[0],
                                'length': part[1],
                                'name': part[2]
                            }
                            for part in chrom
                        ]
                    }
                    for chrom in chromosomes
                ]
            }
        else:
            print(f"No chromosome information found for {species}")
    
    return chromosome_data

def save_to_json(data: Dict, filename: str):
    """Save data to a JSON file with proper formatting."""
    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)

if __name__ == "__main__":
    # Generate chromosome data for all species in ASSEMBLY_MAPPING
    chromosome_data = generate_chromosome_data(list(ASSEMBLY_MAPPING.keys()))
    
    # Save to JSON file
    output_filename = 'chromosome_data.json'
    save_to_json(chromosome_data, output_filename)
    print(f"\nChromosome data has been saved to {output_filename}")